# HW 2: Discrete Heterogeneity

Part 1: Re-estimate the multinomial logit brand choice model with the last brand purchased variable on the Yogurt data, but using a 2 type discrete heterogeneity distribution.  For now put the heterogeneity only on the intercepts. The last assignment had you form an individual specific likelihood function that took an NT by 1 vector and collapsed it to N by 1. For this assignment, first form a likelihood function that is NT by M. Then take the products across T within the first dimension to collapse it into an N by M matrix with the likelihood for each individual if they were each type. Then, take the weighted average of the types over the 2nd dimension to form the individual specific likelihood function which is N by 1.

Part 2: Now try allowing all of the parameters to be type specific (still use 2 types).  Also, try allowing for 4 types but restricting types to vary only by intercepts.

Part 3: Numerically solve for cross price elasticities in the homogenous multinomial logit model and the two type finite support heterogeneity model.  To do so, simulate a small change in the price of brand 1.  Also try it by simulating a small price change for brand 2. Compare elasticities between the homogenous and heterogenous models.

## Results from STATA

```text

```

## Python implementation

In [1]:
# Imports
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.stats import norm
from statsmodels.tools.numdiff import approx_hess

In [2]:
# Import data
df = pd.read_csv("../data/YogurtLong.txt", sep="\t")
print(df.head())

   hh  qty        exp  nopurch  b1  b2  b3  b4    p1    p2    p3    p4  f1  \
0   1    2  40.900002        0   0   0   0   1  0.11  0.08  0.06  0.08   0   
1   1    0   8.830000        1   0   0   0   0  0.11  0.09  0.05  0.08   0   
2   1    0   3.880000        1   0   0   0   0  0.13  0.09  0.05  0.08   0   
3   1    0   0.780000        1   0   0   0   0  0.13  0.09  0.05  0.08   0   
4   1    0  39.240002        1   0   0   0   0  0.13  0.09  0.05  0.08   0   

   f2  f3  f4  tripnum  
0   0   0   0        1  
1   0   0   0        2  
2   0   0   0        3  
3   0   0   0        4  
4   0   0   0        5  


In [3]:
# Convert df to long (each row is a hh-trip-brand choice occasion)
df_long = (
    pd.wide_to_long(
        df,
        stubnames=["b", "p", "f"],
        i=["hh", "tripnum"],
        j="brand",
        suffix=r"\d+"
    )
    .reset_index()
    .sort_values(["hh", "tripnum", "brand"])
)
df_long = pd.get_dummies(df_long, columns=["brand"], prefix="brand", dtype=int)
df_long.head()

,hh,tripnum,exp,nopurch,qty,b,p,f,brand_1,brand_2,brand_3,brand_4
0,1,1,40.900002,0,2,0,0.11,0,1,0,0,0
1,1,1,40.900002,0,2,0,0.08,0,0,1,0,0
2,1,1,40.900002,0,2,0,0.06,0,0,0,1,0
3,1,1,40.900002,0,2,1,0.08,0,0,0,0,1
4,1,2,8.830000,1,0,0,0.11,0,1,0,0,0


### I. 2-type discrete heterogeneity on intercepts

Suppose there are $J$ products and $C$ latent classes. We let brand intercepts $\beta_{0cj}$ ($j=1,\ldots,J$) be class-specific and $\tilde{k}$ other covariates $\beta_1, \ldots, \beta_{\tilde{k}}$ be common across classes. Then, class $c$ has the covariate vector $\vec\beta_c = [\beta_{0c1}, \ldots, \beta_{0cJ}, \beta_1, \cdots, \beta_{\tilde{k}}]^\intercal \in \mathbb{R}^{k}$ where $k=J+\tilde{k}$ is the number of parameters in each covariate vector.

In the following implementation, I create a matrix $B \in \mathbb{R}^{k \times C}$ that stores the covariate vector of class $c$ as its $c$th column (note that $B$ stores all the covariate coefficients to be estimated):
$$
B = \begin{bmatrix} \vec\beta_1 & \cdots & \vec\beta_C \end{bmatrix} = 
\begin{bmatrix} 
\beta_{011} & \cdots & \beta_{0C1} \\
\vdots & \ddots & \vdots \\
\beta_{01J} & \cdots & \beta_{0CJ} \\
\beta_1 & \cdots & \beta_1 \\ 
\vdots & \ddots & \vdots \\
\beta_{\tilde{k}} & \cdots & \beta_{\tilde{k}} \\ 
\end{bmatrix}
$$
Recall that $X \in \mathbb{R}^{NTJ \times k}$ is the design matrix ($NT$ is the number of household-trips made, or equivalently, the number of choice occasions observed). Then $V = XB \in \mathbb{R}^{NTJ \times C}$ stores the utility $v_{itj\mid c}$ of household $i$ in trip $t$ choosing product $j$ if they were of class $c$:
$$
V = \begin{bmatrix}
v_{111\mid 1} & \cdots & v_{111\mid C} \\
\vdots & \ddots & \vdots \\
v_{11J\mid 1} & \cdots & v_{11J\mid C} \\
v_{121\mid 1} & \cdots & v_{121\mid C} \\
\vdots & \ddots & \vdots \\
v_{itj\mid 1} & \cdots & v_{itj\mid C} \\
\vdots & \ddots & \vdots \\
v_{NTJ\mid 1} & \cdots & v_{NTJ\mid C} \\
\end{bmatrix}
$$
We desire an $N \times C$ matrix that contains the likelihoods of each household choices as if they were of each class. To do so, we first reshape $V$ to shape $NT \times J \times C$ and collapse across the second dimension using $1+\sum_{j=1}^J \exp(v_{itj})$. Call the resulting matrix $L_\text{den} \in \mathbb{R}^{NT \times C}$—this is the denominator of the MNL formula. Letting $Y \in \mathbb{R}^{NTJ}$ be the binary vector of whether product $j$ was chosen for each row $itj$, form $\tilde{Y} \in \mathbb{R}^{NTJ \times C}$ that simply copies $Y$ in its $C$ columns. Then, $V \odot \tilde{Y}$ (element-wise matrix multiplication) and reshaping to shape $NT \times J \times C$ before collapsing across the second dimension with a regular sum returns a matrix containing the utility obtained by household $i$ in trip $t$ due to their observed choice, if they were of class $c$. Exponentiating each value gives us $L_\text{num} \in \mathbb{R}^{NT \times C}$—the numerator of the MNL formula. Element-wise division of $L_\text{num}$ over $L_\text{den}$ creates $L^{(3)} \in \mathbb{R}^{NT \times C}$, where $L^{(3)}_{itc}$ is the probability of observing household $i$ choose the product they chose in trip $t$, if they were of class $c$. Taking the product over all the trips $t$ made by household $i$ gives $L^{(2)} \in \mathbb{R}^{N \times C}$: $L^{(2)}_{hc} = \prod_t L^{(3)}_{itc}$.

Now let $\mu=[\mu_1, \ldots, \mu_C]$ be the proportions of each class ($\mu_c \geq 0, \sum_c \mu_c = 1$). To avoid the non-negativity and normalization constraints, we estimate $\{\tilde\mu_1, \ldots, \tilde\mu_C\}$ where $\tilde\mu_c \in \mathbb{R}$ for $c = 2, \ldots, C$ and $\tilde\mu_1=0$. Then, we recover $\mu_c = \frac{\exp(\tilde\mu_c)}{\sum_c \exp(\tilde\mu_c)}$. To obtain the likelihoods of each household, we compute $L^{(1)} = L^{(2)}\mu \in \mathbb{R}^{N}$, which takes a $\mu$-weighted sum over classes.

Finally, we use maximum likelihood estimation on $\sum_{i=1}^N \log L^{(1)}_i$ to obtain parameter estimates for $B$ and $\mu$.

In [4]:
def _unpack_theta(theta, k, n_classes):
    """Unpack theta into B (k x C) and free mu parameters (length C-1)."""
    theta = np.asarray(theta).reshape(-1)
    beta_size = k * n_classes
    expected = beta_size + max(n_classes - 1, 0)
    if theta.size != expected:
        raise ValueError(f"theta must have length {expected}, got {theta.size}")

    B = theta[:beta_size].reshape(k, n_classes, order="F")
    mu_free = theta[beta_size:]  # C-1 free params, with class-1 normalized to 0
    return B, mu_free


def loglik(theta, X, Y, hh_ids, n_alts, n_classes):
    """Negative log-likelihood for finite-mixture MNL with outside option."""
    X = np.asarray(X)
    Y = np.asarray(Y).reshape(-1)
    hh_ids = np.asarray(hh_ids).reshape(-1)

    n, k = X.shape
    n_choices = n // n_alts  # NT

    B, mu_free = _unpack_theta(theta, k, n_classes)

    # Utility tensor: (NT, J, C)
    V = (X @ B).reshape(n_choices, n_alts, n_classes, order="C")

    # Stable log denominator including outside option (utility 0)
    v_max = np.maximum(0.0, V.max(axis=1))
    exp_shifted = np.exp(V - v_max[:, None, :])
    log_denom = v_max + np.log(np.exp(-v_max) + exp_shifted.sum(axis=1))  # (NT, C)

    # Chosen utility per (choice occasion, class)
    Y3 = Y.reshape(n_choices, n_alts, 1, order="C")
    chosen_v = np.multiply(Y3, V).sum(axis=1)  # (NT, C)

    # log L3 per (choice occasion, class)
    log_L3 = chosen_v - log_denom

    # log L2: sum log probabilities over trips within household
    hh_trip = hh_ids.reshape(n_choices, n_alts, order="C")[:, 0]
    _, hh_index = np.unique(hh_trip, return_inverse=True)
    n_households = hh_index.max() + 1
    log_L2 = np.zeros((n_households, n_classes))
    for c in range(n_classes):
        np.add.at(log_L2[:, c], hh_index, log_L3[:, c])

    # Identifying normalization: mu_raw[0] = 0, estimate only C-1 free components
    mu_raw = np.concatenate(([0.0], mu_free)) if n_classes > 1 else np.array([0.0])
    log_mu = mu_raw - logsumexp(mu_raw)

    # log L1_h = logsumexp_c( log L2_hc + log mu_c )
    log_L1 = logsumexp(log_L2 + log_mu[None, :], axis=1)

    return -np.sum(log_L1)


def score_matrix(beta, X, Y, n_alts):
    """
    Returns choice-occasion score vectors s_t(beta) = d log L_t / d beta.
    Shape: (n_choices, k).
    """
    X = np.asarray(X)
    Y = np.asarray(Y).reshape(-1)
    beta = np.asarray(beta).reshape(-1)

    n, k = X.shape
    n_choices = n // n_alts

    # Reshape long format into (choice occasion, alternative, regressor)
    X3 = X.reshape(n_choices, n_alts, k, order="C")
    Y2 = Y.reshape(n_choices, n_alts, order="C")

    # Softmax probabilities including outside option with normalized utility 0
    V = X3 @ beta                                      # (n_choices, n_alts)
    V_with_outside = np.hstack([np.zeros((n_choices, 1)), V])
    P_with_outside = np.exp(V_with_outside - logsumexp(V_with_outside, axis=1, keepdims=True))
    P = P_with_outside[:, 1:]                          # drop outside option prob

    # Score per choice occasion: sum_j (y_tj - p_tj) x_tj
    S = np.einsum("ta,tak->tk", (Y2 - P), X3)
    return S


def print_mle_output(output, covariate_vars):
    """Prints MLE results in aligned tables, separated by class."""
    B_hat = np.asarray(output['opt_beta'])
    n_classes = B_hat.shape[1]
    k = B_hat.shape[0]

    se_theta = np.asarray(output['se_theta'])
    ci_theta = np.asarray(output['ci_theta'])
    beta_size = k * n_classes

    se_beta = se_theta[:beta_size].reshape(k, n_classes, order="F")
    ci_beta = ci_theta[:beta_size].reshape(k, n_classes, 2, order="F")

    print(output.get('message', ''))
    print(f"Log-likelihood: {output['opt_ll']:.4f}")
    if n_classes > 1:
        print(f"Class probs: {np.round(output['opt_mu_prob'], 6)}")
    print()

    widths = (12, 10, 10, 10, 10)
    header_top = f"{'Coefficient':^{widths[0]}} | {'Estimate':^{widths[1]}} | {'Std. Err.':^{widths[2]}} | {'Confidence Interval':^{widths[3] + widths[4] + 3}}"
    divider = "-+-".join("-" * w for w in widths)

    for c in range(n_classes):
        print(f"Class {c + 1}")
        print(header_top)
        print(divider)
        for i, name in enumerate(covariate_vars):
            row = " | ".join([
                f"{name:<{widths[0]}}"
                f"{B_hat[i, c]:>{widths[1]}.6f}"
                f"{se_beta[i, c]:>{widths[2]}.7f}"
                f"{ci_beta[i, c, 0]:>{widths[3]}.5f}"
                f"{ci_beta[i, c, 1]:>{widths[4]}.5f}"
            ])
            print(row)
        print()


def mle_estimator(df,
                  choice_var,
                  covariate_vars,
                  individual_var,
                  n_alts,
                  n_classes=1,
                  beta_init=None,
                  mu_init=None,
                  ci_alpha=0.05,
                  robust_se=False,
                  method='BFGS'):
    """
    Estimates the multinomial logit model using maximum likelihood estimation.
    
    Parameters    
    ----------
    df : DataFrame
        The dataset to be used for estimation.
    choice_var : str
        The column name of the choice variable.
    covariate_vars : list
        A list of column names for the covariates.
    individual_var : str
        The column name of the individual ID variable.
    n_alts : int
        Number of alternatives.
    beta_init : array, optional
        Initial guess for coefficients. If None, defaults to zeros.
    ci_alpha : float
        Significance level for confidence intervals.
    robust_se : bool
        Whether to compute robust standard errors.
    method : str
        Optimization method.

    Returns    
    -------
    Dictionary containing:
        - optimized log-likelihood 
        - coefficient estimates
        - standard errors
        - confidence intervals for coefficients
    """
    # Prepare data
    Y = df[[choice_var]].to_numpy()
    X = df[covariate_vars].to_numpy()
    individual_ids = df[individual_var].to_numpy()
    k = X.shape[1]

    if beta_init is None:
        beta_init = np.zeros((k, n_classes))
    else:
        beta_init = np.asarray(beta_init).reshape(k, n_classes, order="F")

    # mu_init can be length C (raw levels) or C-1 (free levels). We optimize only C-1.
    if n_classes == 1:
        mu_free_init = np.array([])
    else:
        if mu_init is None:
            mu_free_init = np.zeros(n_classes - 1)

    theta_init = np.concatenate([beta_init.reshape(-1, order="F"), mu_free_init])

    result = minimize(
        fun=loglik,
        x0=theta_init,
        args=(X, Y, individual_ids, n_alts, n_classes),
        method=method,
    )

    opt_ll = -result.fun
    opt_theta = result.x
    B_hat, mu_free_hat = _unpack_theta(opt_theta, k, n_classes)
    mu_raw_hat = np.concatenate(([0.0], mu_free_hat)) if n_classes > 1 else np.array([0.0])
    mu_prob_hat = np.exp(mu_raw_hat - logsumexp(mu_raw_hat))

    H = approx_hess(opt_theta, loglik, args=(X, Y, individual_ids, n_alts, n_classes))
    H_inv = np.linalg.pinv(H)
    se_theta = np.sqrt(np.diag(H_inv))

    z_score = norm.ppf(1 - ci_alpha / 2)
    ci_theta = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_theta, se_theta)]

    output = {
        'opt_ll': opt_ll,
        'opt_theta': opt_theta,
        'opt_beta': B_hat,
        'opt_mu_raw': mu_raw_hat,
        'opt_mu_prob': mu_prob_hat,
        'se_theta': se_theta,
        'ci_theta': ci_theta,
        'success': result.success,
        'message': result.message,
    }

    if robust_se:
        # J = E[s_t s_t'] estimated by sample average of outer products of scores.
        S = score_matrix(opt_beta, X, Y, n_alts)       # (n_choices, k)
        n_choices = S.shape[0]
        J = S.T @ S

        # Sandwich variance: H^{-1} J H^{-1}
        V_robust = H_inv @ J @ H_inv
        se_beta_r = np.sqrt(np.diag(V_robust))
        ci_r = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_beta, se_beta_r)]

        output['se_beta'] = se_beta_r
        output['ci'] = ci_r
    
    print_mle_output(output, covariate_vars)

    return output

In [5]:
df_long["prev_purch"] = df_long.groupby("hh")["b"].shift(4).fillna(0).astype(int)
df_long.head()

hh_ids = df[['hh']].to_numpy()  # trip-level household IDs (NT,)
results = mle_estimator(
    df_long,
    choice_var='b',
    covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
    individual_var='hh',
    n_alts=4,
    n_classes=2,
    robust_se=False,
    method='BFGS',
)

Desired error not necessarily achieved due to precision loss.
Log-likelihood: -7035.0154
Class probs: [0. 1.]

Class 1
Coefficient  |  Estimate  | Std. Err.  |   Confidence Interval  
-------------+------------+------------+------------+-----------
brand_1      -3.794691 0.0000000  -3.79469  -3.79469
brand_2      -5.394110 0.0000000  -5.39411  -5.39411
brand_3      -3.529072 0.0000000  -3.52907  -3.52907
brand_4      -4.720354 0.0000000  -4.72035  -4.72035
f            20.621718 0.0000000  20.62172  20.62172
p            -7.424088 0.0000000  -7.42409  -7.42409
prev_purch   -3.122372 0.0000000  -3.12237  -3.12237

Class 2
Coefficient  |  Estimate  | Std. Err.  |   Confidence Interval  
-------------+------------+------------+------------+-----------
brand_1       0.562165 0.1680918   0.23271   0.89162
brand_2      -0.144811 0.1342447  -0.40793   0.11830
brand_3      -3.361736 0.1444109  -3.64478  -3.07870
brand_4      -0.632834 0.1340350  -0.89554  -0.37013
f             0.353774 0.0968